# 01. Очистка данных и временные признаки

Ноутбук читает исходный файл `data/raw/Last.fm_data.csv`, чистит данные, строит временные признаки и сохраняет итоговую таблицу `data/interim/lastfm_time_features.csv`.

## 1. Импорт библиотек

In [1]:
from pathlib import Path

import pandas as pd

## 2. Пути проекта и выходной файл

In [2]:
PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

RAW_LASTFM_PATH = PROJECT_ROOT / "data" / "raw" / "Last.fm_data.csv"
TIME_FEATURES_OUTPUT_PATH = PROJECT_ROOT / "data" / "interim" / "lastfm_time_features.csv"

TIME_FEATURES_OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)

RAW_LASTFM_PATH, TIME_FEATURES_OUTPUT_PATH

(PosixPath('/Users/zarmeux/hse/course work 1/course-project-listener-classification/data/raw/Last.fm_data.csv'),
 PosixPath('/Users/zarmeux/hse/course work 1/course-project-listener-classification/data/interim/lastfm_time_features.csv'))

## 3. Константы и словари для преобразований

In [3]:
RAW_COLUMNS = [
    "Username",
    "Artist",
    "Track",
    "Album",
    "Date",
    "Time"
]
TEXT_COLUMNS = [
    "username",
    "artist",
    "track",
    "album",
    "date",
    "time"
]
RENAMED_COLUMNS = dict(zip(RAW_COLUMNS, TEXT_COLUMNS))

MONTH_NUMBER_TO_RU = dict(
    enumerate([
        "Январь",
        "Февраль",
        "Март",
        "Апрель",
        "Май",
        "Июнь",
        "Июль",
        "Август",
        "Сентябрь",
        "Октябрь",
        "Ноябрь",
        "Декабрь",
    ], 1)
)

WEEKDAY_NUMBER_TO_RU = dict(
    enumerate([
        "Понедельник",
        "Вторник",
        "Среда",
        "Четверг",
        "Пятница",
        "Суббота",
        "Воскресенье",
    ])
)

WEEKEND_TO_RU = {
    False: "Будни",
    True:  "Выходные",
}

## 4. Функции очистки данных и построения временных признаков

In [4]:
def clean_lastfm_data(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()

    unnamed_columns = [column for column in df.columns if column.startswith("Unnamed:")]
    if unnamed_columns:
        df = df.drop(columns=unnamed_columns)

    missing_columns = sorted(set(RAW_COLUMNS) - set(df.columns))
    assert len(missing_columns) == 0, (
        f"Missing required columns: {missing_columns}"
    )

    df = df[RAW_COLUMNS].rename(columns=RENAMED_COLUMNS).copy()

    for column in TEXT_COLUMNS:
        df[column] = df[column].astype("string").str.strip()

    df = df[df[TEXT_COLUMNS].notna().all(axis=1)].copy()
    df = df[(df[TEXT_COLUMNS] != "").all(axis=1)].copy()

    df["datetime"] = pd.to_datetime(
        df["date"] + " " + df["time"],
        format="%d %b %Y %H:%M",
        errors="coerce",
    )

    df = df[df["datetime"].notna()].copy()
    df = df.drop_duplicates().reset_index(drop=True)
    return df


def get_day_part(hour: int) -> str:
    if 6 <= hour <= 11:
        return "Утро"
    if 12 <= hour <= 17:
        return "День"
    if 18 <= hour <= 23:
        return "Вечер"
    return "Ночь"


def build_time_features(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    df["datetime"] = pd.to_datetime(df["datetime"])

    df["date_only"]      = df["datetime"].dt.normalize()
    df["year"]           = df["datetime"].dt.year.astype(int)
    df["month_number"]   = df["datetime"].dt.month.astype(int)
    df["month"]          = df["month_number"].map(MONTH_NUMBER_TO_RU)
    df["weekday_number"] = df["datetime"].dt.weekday.astype(int)
    df["weekday"]        = df["weekday_number"].map(WEEKDAY_NUMBER_TO_RU)
    df["hour"]           = df["datetime"].dt.hour.astype(int)
    df["day_part"]       = df["hour"].map(get_day_part)
    df["is_weekend"]     = df["weekday_number"].isin([5, 6])
    df["weekend"]        = df["is_weekend"].map(WEEKEND_TO_RU)

    columns = [
        "username",
        "artist",
        "track",
        "album",
        "datetime",
        "date_only",
        "year",
        "month_number",
        "month",
        "weekday_number",
        "weekday",
        "hour",
        "day_part",
        "is_weekend",
        "weekend",
    ]
    return df[columns]

## 5. Загрузка исходных данных и основной расчёт

In [5]:
raw_df = pd.read_csv(RAW_LASTFM_PATH, keep_default_na=False)
print(f"Shape of raw data:            {raw_df.shape}")

clean_df = clean_lastfm_data(raw_df)
dropped_rows = len(raw_df) - len(clean_df)
print(f"Rows dropped during cleaning: {dropped_rows}")
print(f"Shape of clean data:          {clean_df.shape}")

time_features_df = build_time_features(clean_df)
print(f"Shape of time features data:  {time_features_df.shape}")

Shape of raw data:            (166153, 7)
Rows dropped during cleaning: 0
Shape of clean data:          (166153, 7)
Shape of time features data:  (166153, 15)


## 6. Проверка структуры и сохранение результата

In [6]:
required_columns = {
    "username",
    "artist",
    "track",
    "album",
    "datetime",
    "date_only",
    "year",
    "month_number",
    "month",
    "weekday_number",
    "weekday",
    "hour",
    "day_part",
    "is_weekend",
    "weekend",
}

assert set(time_features_df.columns) == required_columns
assert not time_features_df.empty
assert not time_features_df[["username", "artist", "track", "album"]].isna().any().any()
assert set(time_features_df["day_part"].unique()) <= {"Ночь", "Утро", "День", "Вечер"}
assert set(time_features_df["weekend"].unique()) <= {"Будни", "Выходные"}

time_features_df.to_csv(TIME_FEATURES_OUTPUT_PATH, index=False)
print(f"Saved file: {TIME_FEATURES_OUTPUT_PATH}")

Saved file: /Users/zarmeux/hse/course work 1/course-project-listener-classification/data/interim/lastfm_time_features.csv


## 7. Предварительный просмотр итоговой таблицы

In [7]:
time_features_df.head()

,username,artist,track,album,datetime,date_only,year,month_number,month,weekday_number,weekday,hour,day_part,is_weekend,weekend
0,Babs_05,Isobel Campbell,The Circus Is Leaving Town,Ballad of the Broken Seas,2021-01-31 23:36:00,2021-01-31,2021,1,Январь,6,Воскресенье,23,Вечер,True,Выходные
1,Babs_05,Isobel Campbell,Dusty Wreath,Ballad of the Broken Seas,2021-01-31 23:32:00,2021-01-31,2021,1,Январь,6,Воскресенье,23,Вечер,True,Выходные
2,Babs_05,Isobel Campbell,Honey Child What Can I Do?,Ballad of the Broken Seas,2021-01-31 23:28:00,2021-01-31,2021,1,Январь,6,Воскресенье,23,Вечер,True,Выходные
3,Babs_05,Isobel Campbell,It's Hard To Kill A Bad Thing,Ballad of the Broken Seas,2021-01-31 23:25:00,2021-01-31,2021,1,Январь,6,Воскресенье,23,Вечер,True,Выходные
4,Babs_05,Isobel Campbell,Saturday's Gone,Ballad of the Broken Seas,2021-01-31 23:21:00,2021-01-31,2021,1,Январь,6,Воскресенье,23,Вечер,True,Выходные
